# Nomobug Calendar Events

Beginner-friendly CP2 notebook. Run the cells from top to bottom.

It loads **all populated rows** but only displays small previews so the notebook stays responsive. Credentials and real source IDs stay in the ignored local `secrets/` and `data/profiles/` folders.

## 1. Import pandas and connect to the existing local Google OAuth files

In [ ]:
from pathlib import Path
from zoneinfo import ZoneInfo
import datetime as dt
import json
import pandas as pd
from IPython.display import display
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

token_path = project_root / "secrets" / "google_token.json"
metadata_path = project_root / "data" / "profiles" / "google_source_metadata.json"

print("Project folder:", project_root)
print("Token found:", token_path.exists())

In [ ]:
scopes = [
    "https://www.googleapis.com/auth/spreadsheets.readonly",
    "https://www.googleapis.com/auth/calendar.readonly",
]

credentials = Credentials.from_authorized_user_file(str(token_path), scopes)
calendar_service = build("calendar", "v3", credentials=credentials, cache_discovery=False)
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))

calendars = [item for item in metadata["calendars"] if item.get("target_name_match")]
calendar_names_df = pd.DataFrame({"calendar_name": [item["summary"] for item in calendars]})
display(calendar_names_df)

## 2. Choose a complete finite date window

The extraction uses pagination so it includes every event occurrence returned within the window.

In [ ]:
start_date = dt.date(2000, 1, 1)
end_date = dt.date.today()
timezone_name = "Asia/Kuala_Lumpur"
local_timezone = ZoneInfo(timezone_name)

time_min = dt.datetime.combine(start_date, dt.time.min, tzinfo=local_timezone).isoformat()
time_max = dt.datetime.combine(end_date + dt.timedelta(days=1), dt.time.min, tzinfo=local_timezone).isoformat()

print("From:", start_date)
print("Through:", end_date)

## 3. Load every event from all six Calendars into one DataFrame

In [ ]:
event_rows = []

for calendar in calendars:
    calendar_name = calendar["summary"]
    calendar_id = calendar["id"]
    page_token = None

    while True:
        response = calendar_service.events().list(
            calendarId=calendar_id,
            timeMin=time_min,
            timeMax=time_max,
            timeZone=timezone_name,
            maxResults=2500,
            singleEvents=True,
            orderBy="startTime",
            showDeleted=False,
            pageToken=page_token,
        ).execute()

        for event in response.get("items", []):
            start = event.get("start", {})
            end = event.get("end", {})
            event_rows.append({
                "calendar_name": calendar_name,
                "calendar_id": calendar_id,
                "event_id": event.get("id"),
                "status": event.get("status"),
                "start": start.get("dateTime") or start.get("date"),
                "end": end.get("dateTime") or end.get("date"),
                "is_all_day": bool(start.get("date") and not start.get("dateTime")),
                "summary": event.get("summary"),
                "description": event.get("description"),
                "location": event.get("location"),
                "recurring_event_id": event.get("recurringEventId"),
            })

        page_token = response.get("nextPageToken")
        if not page_token:
            break

events_df = pd.DataFrame(event_rows)

print("All loaded events:", len(events_df))
print("Exact duplicate rows:", int(events_df.duplicated().sum()))
display(events_df.head())

## 4. Summarise coverage without displaying customer event text

In [ ]:
events_df["start_parsed"] = pd.to_datetime(events_df["start"], errors="coerce", utc=True)

calendar_summary_df = events_df.groupby("calendar_name", dropna=False).agg(
    event_rows=("event_id", "size"),
    distinct_event_ids=("event_id", "nunique"),
    earliest_start_utc=("start_parsed", "min"),
    latest_start_utc=("start_parsed", "max"),
    missing_summary=("summary", lambda values: values.isna().sum()),
    missing_description=("description", lambda values: values.isna().sum()),
).reset_index()

display(calendar_summary_df)

## 5. Create a column-quality profile

In [ ]:
calendar_profile_df = pd.DataFrame({
    "column": events_df.columns,
    "non_null_count": events_df.notna().sum().values,
    "missing_count": events_df.isna().sum().values,
    "missing_percent": (events_df.isna().mean() * 100).round(2).values,
    "distinct_count": events_df.nunique(dropna=True).values,
})

display(calendar_profile_df)

## 6. Save every loaded event locally

Calendar titles, descriptions and locations may contain personal data. These files remain ignored by Git.

In [ ]:
raw_folder = project_root / "data" / "raw" / "calendar"
profile_folder = project_root / "data" / "profiles" / "calendar"
raw_folder.mkdir(parents=True, exist_ok=True)
profile_folder.mkdir(parents=True, exist_ok=True)

events_df.drop(columns=["start_parsed"]).to_csv(
    raw_folder / "nomobug_calendar_events_all_history.csv",
    index=False,
)
calendar_summary_df.to_csv(profile_folder / "calendar_summary.csv", index=False)
calendar_profile_df.to_csv(profile_folder / "calendar_column_profile.csv", index=False)

print("Saved every event to:", raw_folder)
print("Saved profiles to:", profile_folder)